# Demo 4 - Build the resilient pool

## Scenario

Routing is a **per-call decision**, not a deployment-time choice. For every admitted request, APIM selects a pool member by **priority, then weight, then health**. This demo builds two priority-1 PTU members -- East at weight 2 and Central at weight 1 -- plus a priority-2 PAYG spillover member.

Keep the **same model and version** on every load-balanced and failover backend. A mixed pool can succeed while silently changing model behavior.


## Isolation

Demo 4 creates only `demo4-*` resources on the existing APIM instance used by Demos 1-3. Every ARM write is a PUT/upsert, so running this notebook twice does not duplicate resources.


In [ ]:
import sys
sys.path.append("..")

import re
import time
from collections import Counter

import requests

from shared import auth, config, apim, display

cfg = config.load_config(interactive=True)
config.validate_config(cfg)
cfg = config.ensure_resilient_pool_config(cfg, interactive=True)
config.validate_resilient_pool_config(cfg)

DEMO_API_ID = "demo4-resilient-pool-api"
DEMO_PRODUCT_ID = "demo4-resilient-pool"
DEMO_SUBSCRIPTION_ID = "demo4-resilient-pool-sub"
DEMO_PATH = "demo4-resilient-pool"
DEMO_NAMED_VALUE_AOAI_KEY = "demo4-aoai-key"
POOL_ID = "demo4-aoai-pool"
POOL_MEMBERS = [
    ("demo4-ptu-east", cfg.demo4_ptu_east_endpoint, 1, 2),
    ("demo4-ptu-central", cfg.demo4_ptu_central_endpoint, 1, 1),
    ("demo4-payg", cfg.demo4_payg_endpoint, 2, 1),
]
API_STYLE = cfg.aoai_api_style
SIMULATED_TOPOLOGY = len({endpoint for _, endpoint, _, _ in POOL_MEMBERS}) == 1

print(f"Demo 4 pool: {POOL_ID}; API style: {API_STYLE}")
if SIMULATED_TOPOLOGY:
    display.banner(
        "Topology is simulated: all three logical members use AOAI_ENDPOINT. "
        "The pool routing and circuit-breaker mechanics are identical.", kind="warning"
    )


## Preflight checks

The pool and circuit-breaker backend contract uses ARM API version `2024-06-01-preview`. It requires APIM Basic v2, Standard v2, Premium v2, or classic Standard/Premium.


In [ ]:
display.header("Preflight checks")
preflight_rows = []
try:
    service = apim.get_service(cfg.subscription_id, cfg.resource_group, cfg.apim_name)
    sku = service.get("sku", {}).get("name", "")
    supported_skus = {"BasicV2", "StandardV2", "PremiumV2", "Standard", "Premium"}
    preflight_rows.append({
        "check": "APIM instance reachable", "status": "PASS",
        "detail": f"{cfg.apim_name} reachable; SKU={sku}", "remediation": "",
    })
    preflight_rows.append({
        "check": "APIM SKU supports pools and circuit breakers",
        "status": "PASS" if sku in supported_skus else "FAIL",
        "detail": sku or "SKU was not returned",
        "remediation": "Use Basic v2, Standard v2, Premium v2, or classic Standard/Premium.",
    })
except Exception as exc:
    preflight_rows.append({"check": "APIM instance reachable", "status": "FAIL", "detail": str(exc), "remediation": "Verify APIM_RESOURCE_GROUP and APIM_NAME."})

aoai_ok = bool(cfg.aoai_endpoint and cfg.aoai_deployment)
preflight_rows.append({"check": "Azure OpenAI values present", "status": "PASS" if aoai_ok else "FAIL", "detail": f"endpoint={cfg.aoai_endpoint}, deployment={cfg.aoai_deployment}", "remediation": "Complete Demo 1 or set AOAI_ENDPOINT and AOAI_DEPLOYMENT."})
try:
    config.validate_resilient_pool_config(cfg)
    pool_ok, pool_detail = True, f"deployment={cfg.demo4_ptu_east_deployment} across all members"
except ValueError as exc:
    pool_ok, pool_detail = False, str(exc)
preflight_rows.append({"check": "Same model/version across pool members", "status": "PASS" if pool_ok else "FAIL", "detail": pool_detail, "remediation": "Set every DEMO4_*_DEPLOYMENT to the same AOAI deployment."})
_ = display.show_table(preflight_rows, columns=["check", "status", "detail", "remediation"])
if any(row["status"] == "FAIL" for row in preflight_rows):
    raise RuntimeError("Fix the failed preflight checks before configuring Demo 4.")


## Configure (policy apply)

Create the three individual backends with identical circuit-breaker rules, then create the pool. The breaker trips after two 429/5xx responses in one minute, remains open for one minute, and honors an origin `Retry-After`.


In [ ]:
display.header("Creating resilient backends and pool")
CIRCUIT_BREAKER = {
    "rules": [{
        "name": "demo4-transient-failures",
        "failureCondition": {
            "count": 2,
            "interval": "PT1M",
            "statusCodeRanges": [{"min": 429, "max": 429}, {"min": 500, "max": 599}],
        },
        "tripDuration": "PT1M",
        "acceptRetryAfter": True,
    }]
}
for backend_id, endpoint, _, _ in POOL_MEMBERS:
    apim.ensure_backend(
        cfg.subscription_id, cfg.resource_group, cfg.apim_name, backend_id, endpoint,
        description=f"Demo 4 pool member {backend_id}", circuit_breaker=CIRCUIT_BREAKER,
    )
apim.ensure_backend_pool(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name, POOL_ID,
    [{"id": backend_id, "priority": priority, "weight": weight}
     for backend_id, _, priority, weight in POOL_MEMBERS],
    description="Demo 4 priority/weight resilient Azure OpenAI pool",
)
display.banner("Three backends and the priority/weight pool are ensured with ARM 2024-06-01-preview.", kind="success")


In [ ]:
display.header("Creating API, product, subscription, and optional key")
apim.ensure_api(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_API_ID,
                "Demo 4 - Resilient backend pool", DEMO_PATH, cfg.aoai_endpoint.rstrip("/"))
OPERATION_URL_TEMPLATE = "/openai/v1/chat/completions" if API_STYLE == "v1" else f"/openai/deployments/{cfg.aoai_deployment}/chat/completions"
apim.ensure_operation(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_API_ID,
                      "chat-completions", "Chat Completions", "POST", OPERATION_URL_TEMPLATE)
apim.ensure_product(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_PRODUCT_ID,
                    "Demo4-Resilient-Pool", "Isolated product for Demo 4 resilient routing.", True, "published")
apim.ensure_product_api_link(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_PRODUCT_ID, DEMO_API_ID)
apim.ensure_subscription(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_SUBSCRIPTION_ID,
                         "Demo4-Resilient-Pool-Subscription", f"/products/{DEMO_PRODUCT_ID}")
if cfg.aoai_key:
    apim.ensure_named_value(cfg.subscription_id, cfg.resource_group, cfg.apim_name,
                            DEMO_NAMED_VALUE_AOAI_KEY, DEMO_NAMED_VALUE_AOAI_KEY, cfg.aoai_key, secret=True)
display.banner("Demo 4 API, product, subscription, and optional named value ensured.", kind="success")


In [ ]:
with open("../policies/demo4-resilient-pool.xml", encoding="utf-8-sig") as f:
    policy_xml = f.read()
mi_block = re.compile(r"\s*<authentication-managed-identity\b[^>]*/>")
if cfg.aoai_key:
    policy_xml, replaced = mi_block.subn(
        '\n    <set-header name="api-key" exists-action="override">\n      <value>{{demo4-aoai-key}}</value>\n    </set-header>', policy_xml, count=1)
    if not replaced:
        raise ValueError("Could not find the managed-identity block in the policy XML.")
    display.banner("AOAI_KEY supplied: policy uses the demo4-aoai-key named value.", kind="info")
else:
    display.banner("No AOAI_KEY: policy uses the APIM managed identity.", kind="info")
apim.set_api_policy(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_API_ID, policy_xml)
display.banner("API-scope policy applied. The one set-backend-service line targets the pool.", kind="success")


## Baseline

Send one request through the pool. `x-demo4-routing` truthfully confirms policy routing to the pool, but APIM policy context does not expose the selected member id. Use gateway diagnostics to observe an individual serving backend.


In [ ]:
GATEWAY_URL = apim.get_gateway_url(cfg.subscription_id, cfg.resource_group, cfg.apim_name)
SUBSCRIPTION_KEY = apim.get_subscription_key(cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_SUBSCRIPTION_ID)
def call_pool(prompt):
    url = f"{GATEWAY_URL}/{DEMO_PATH}" + ("/openai/v1/chat/completions" if API_STYLE == "v1" else f"/openai/deployments/{cfg.aoai_deployment}/chat/completions")
    body = {"messages": [{"role": "user", "content": prompt}], "max_tokens": 40}
    if API_STYLE == "v1": body["model"] = cfg.aoai_deployment
    start = time.time()
    response = requests.post(url, headers={"Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY, "Content-Type": "application/json"}, params={} if API_STYLE == "v1" else {"api-version": cfg.aoai_api_version}, json=body, timeout=90)
    return {"status": response.status_code, "latency_ms": round((time.time()-start)*1000, 1), "routing": response.headers.get("x-demo4-routing"), "body": response.text[:500]}
baseline = call_pool("Reply with one short greeting.")
_ = display.show_table([baseline])


## Demonstration: weighted distribution

Fire 30 requests. The expected selection is approximately 2:1 East:Central among healthy priority-1 members. The response cannot safely claim a member id, so this cell reports only observable response headers and labels any distribution as **gateway-diagnostics required**, never fabricated.


In [ ]:
N = 30
weighted_results = [call_pool("Reply with the word pool.") for _ in range(N)]
observed_headers = Counter(result["routing"] or "not exposed" for result in weighted_results)
_ = display.show_table([{"observable_signal": key, "calls": value} for key, value in observed_headers.items()])
display.banner(
    "Member identity is not exposed by APIM policy context, so an East:Central count is not inferred here. "
    "Use APIM gateway diagnostics/backend logs to group the selected backend id; then expect roughly 2:1. "
    "The x-demo4-routing header proves only that the pool policy ran.", kind="info"
)


## Demonstration: circuit breaker and priority spillover

Temporarily replace both priority-1 backend URLs with an invalid origin, make enough calls to trip their 5xx breaker, then observe successful traffic from priority-2 PAYG. This is reversible: the `finally` block restores both original URLs even if the demonstration call fails.


In [ ]:
priority_one = POOL_MEMBERS[:2]
FAILING_ORIGIN = "https://demo4-invalid-origin.invalid"
try:
    for backend_id, _, _, _ in priority_one:
        apim.ensure_backend(cfg.subscription_id, cfg.resource_group, cfg.apim_name, backend_id,
                            FAILING_ORIGIN, description=f"Temporary Demo 4 failing member {backend_id}", circuit_breaker=CIRCUIT_BREAKER)
    failure_attempts = [call_pool("Trip the temporary breaker.") for _ in range(4)]
    spillover = call_pool("If priority 1 is unhealthy, this should use priority-2 PAYG.")
    _ = display.show_table([{"phase": "trip", **result} for result in failure_attempts] + [{"phase": "spillover", **spillover}])
    display.banner("A 200 spillover response demonstrates PAYG availability; verify member id in gateway diagnostics.", kind="success" if spillover["status"] == 200 else "warning")
finally:
    for backend_id, endpoint, _, _ in priority_one:
        apim.ensure_backend(cfg.subscription_id, cfg.resource_group, cfg.apim_name, backend_id,
                            endpoint, description=f"Demo 4 pool member {backend_id}", circuit_breaker=CIRCUIT_BREAKER)
    display.banner("Priority-1 backend URLs restored. Wait one minute (tripDuration) before demonstrating recovery.", kind="info")

recovery = call_pool("After recovery, reply with one word.")
_ = display.show_table([recovery])


## Retry-After honoring

When a pool member returns 429 with `Retry-After`, `acceptRetryAfter: true` keeps that member out of selection for the origin-supplied interval (rather than only the static `tripDuration`). A real 429 must come from the origin or a controlled test backend; this notebook does not manufacture response evidence.


## Summary

- **Per-call selection:** APIM evaluates priority, then weight, then health. Circuit state removes unhealthy members; PAYG is used only when priority 1 is unavailable.
- **One policy line:** `<set-backend-service backend-id="demo4-aoai-pool" />` replaces bespoke retry and failover logic scattered across clients.
- **Consistency rule:** every pool member must use the same model and version, or routing can silently change model behavior.

Re-running this notebook is safe: every Demo 4 resource is an idempotent ARM PUT.


## Reset / teardown notes

The following **optional, clearly gated** cell removes Demo 1-4 workshop artifacts only. It never deletes the APIM instance. Set `REMOVE_WORKSHOP_ARTIFACTS = True` only after the workshop.


In [ ]:
REMOVE_WORKSHOP_ARTIFACTS = False
if REMOVE_WORKSHOP_ARTIFACTS:
    resources = [
        "apis/demo2-metering-api/diagnostics/applicationinsights",
        "apis/demo1-openai-api", "apis/demo2-metering-api", "apis/demo3-content-safety-api", "apis/demo4-resilient-pool-api",
        "subscriptions/demo1-token-governance-sub", "subscriptions/demo2-metering-sub", "subscriptions/demo3-content-safety-sub", "subscriptions/demo4-resilient-pool-sub",
        "backends/demo1-openai-backend", "backends/demo2-openai-backend", "backends/demo3-openai-backend", "backends/demo3-content-safety-backend", "backends/demo4-aoai-pool", "backends/demo4-ptu-east", "backends/demo4-ptu-central", "backends/demo4-payg",
        "namedValues/demo1-tokens-per-minute", "namedValues/demo1-daily-token-cap", "namedValues/demo1-aoai-key", "namedValues/demo2-aoai-key", "namedValues/demo3-aoai-key", "namedValues/demo3-content-safety-key", "namedValues/demo3-content-safety-blocklist-id", "namedValues/demo3-content-safety-threshold-hate", "namedValues/demo3-content-safety-threshold-selfharm", "namedValues/demo3-content-safety-threshold-sexual", "namedValues/demo3-content-safety-threshold-violence", "namedValues/demo4-aoai-key",
        "loggers/demo2-application-insights", "products/demo1-token-governance-product", "products/demo2-metering", "products/demo3-content-safety", "products/demo4-resilient-pool",
    ]
    deleted = [path for path in resources if apim.delete_apim_resource_if_exists(cfg.subscription_id, cfg.resource_group, cfg.apim_name, path)]
    _ = display.show_table([{"deleted": path} for path in deleted])
    display.banner("Workshop artifacts removed; the APIM instance was left intact.", kind="success")
else:
    display.banner("Teardown is disabled. Set REMOVE_WORKSHOP_ARTIFACTS = True only after the workshop.", kind="info")
